# FreeFine Discovery Run — Account A (Feature/Frequency) — v2

Robust M-HFF patch: insertion no longer depends on the exact upstream comment after `all_intermediate_features.append(sample)`.


## 1. Clone + timer

In [1]:
# ===== C1 clone + clock =====
import time, subprocess
NB_START=time.time()
subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && git clone -q https://github.com/CIawevy/FreeFine.git", shell=True, check=True); print("cloned")

cloned


## 2. Generation environment — identical versions to prior combined-model run

In [2]:
%%bash
# ===== C2 freefine_env (generation) =====
set -e
pip install -q --root-user-action=ignore uv; uv python install 3.10.13
V=/kaggle/temp/freefine_env; PY=$V/bin/python
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/temp/FreeFine
uv pip install --python "$PY" -r requirements.txt || { grep -v '^xformers' requirements.txt>/tmp/r.txt; uv pip install --python "$PY" -r /tmp/r.txt; uv pip install --python "$PY" xformers; }
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70"
"$PY" -c "import torch,diffusers,xformers; print('freefine_env OK',torch.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 61.7 MB/s eta 0:00:00
freefine_env OK 2.1.1+cu121


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.46s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/freefine_env
Activate with: source /kaggle/temp/freefine_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/freefine_env
Resolved 18 packages in 643ms
 Downloaded torchvision
 Downloaded triton
 Downloaded pillow
 Downloaded networkx
 Downloaded numpy
 Downloaded sympy
 Downloaded torch
Prepared 18 packages in 36.82s
Installed 18 packages in 264ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.32.3
 + fsspec==2026.7.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.16.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/temp/freefine_en

## 3. Metric environment — identical versions to prior combined-model run

In [3]:
%%bash
# ===== C3 metric_env (evaluation) =====
set -e
V=/kaggle/temp/metric_env; PY=$V/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null||true; done
echo "metric_env OK"

metric_env OK


Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 633ms
 Downloaded torchaudio
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchvision
 Downloaded nvidia-nvjitlink-cu12
 Downloaded pillow
 Downloaded networkx
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-curand-cu12
 Downloaded triton
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cusolver-cu12
 Downloaded numpy
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded sympy
 Downloaded nvidia-cublas-cu12
 Downloaded torch
Prepared 27 packages in 45.64s
Installed 27 packages in 345ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4

## 4. Reproducibility/metric patches — identical seeded-MD setup

In [4]:
# ===== C4 patch metrics (args.3d, SD-2.1 mirror, SEEDED MD) + model.py start_layer =====
import pathlib, re
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"; mp.write_text(mp.read_text().replace("args.3d","getattr(args,'3d')"))
for f in [mr/"MD"/"mean_distance.py", mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))
md=mr/"MD"/"mean_distance.py"; s=md.read_text()
s=s.replace("all_dist = []","all_dist = []\n    import torch as _st; _st.manual_seed(42); _st.cuda.manual_seed_all(42)",1); md.write_text(s)
mm=pathlib.Path("/kaggle/temp/FreeFine/src/demo/model.py"); g=mm.read_text()
if not re.search(r'^\s*import os\b', g, re.M): g="import os\n"+g
assert "list(range(10, 16))" in g, "layer_idx hardcode missing"
n=g.count("list(range(10, 16))")
g=g.replace("list(range(10, 16))","list(range(int(os.environ.get('FF_START_LAYER','10')), 16))")
mm.write_text(g); print(f"patched metrics + model.py start_layer ({n} sites)")

patched metrics + model.py start_layer (5 sites)


## 5. Parameterize the released 2D inference script

In [5]:
# ===== C5 parametrize inference script =====
import os
P="/kaggle/temp/FreeFine/evaluation/FreeFine"; src=open(f"{P}/freefine_batch_infer_2d.py").read()
src=src.replace("sys.path.append('/data/Hszhu/FreeFine')","sys.path.append('/kaggle/temp/FreeFine')")
src=src.replace('pretrained_model_path = "/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/"','pretrained_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"')
old=('        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n        obj_label = ""\n        ori_mask = read_and_resize_mask(ori_mask_path)\n')
new=('        ori_mask = read_and_resize_mask(ori_mask_path)\n'
     '        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
     '        obj_label = (case.get("obj_label","") if os.environ.get("FF_USE_PROMPT")=="1" else "")\n')
assert old in src, "ori_mask/obj_label block mismatch"; src=src.replace(old,new,1)
src=src.replace('"guidance_scale": 7.5,','"guidance_scale": float(os.environ.get("FF_GUIDANCE","7.5")),')
src=src.replace('"start_step": 35,','"start_step": int(os.environ.get("FF_START_STEP","35")),')
src=src.replace('dataset_json = osp.join(dst_base, "annotations_2d.json")','dataset_json = os.environ.get("FF_SUBSET_JSON", osp.join(dst_base,"annotations_2d.json"))')
src=src.replace('dst_gen_dir = osp.join(dst_base, "Geo-Bench-2D/Gen_results_FreeFine_2d")','dst_gen_dir = os.environ.get("FF_OUT_DIR", osp.join(dst_base,"Geo-Bench-2D/Gen_results_FreeFine_2d"))')
src=src.replace('base_dir = "/data/Hszhu/dataset/GeoBenchMeta/"','base_dir = "/kaggle/temp/GeoBenchMeta"')
open(f"{P}/freefine_sweep_2d.py","w").write(src)
assert all(x in src for x in ["FF_USE_PROMPT","FF_GUIDANCE","FF_START_STEP"]), "parametrize failed"
print("parametrized script written")

parametrized script written


## 6. Account A method patch — R1 + M-HFF / full-feature ablation

In [6]:
# ===== C5A patch: R1 router + masked high-frequency decoder-feature fusion (M-HFF) =====
# FIXED: the forward_sampling UNet anchor is now matched by regex and the replacement block
# is re-indented to the real upstream indentation (16 spaces), instead of assuming 12.
import os, re, pathlib

S="/kaggle/temp/FreeFine/evaluation/FreeFine/freefine_sweep_2d.py"
M="/kaggle/temp/FreeFine/src/demo/model.py"
A="/kaggle/temp/FreeFine/src/utils/attention.py"

s=open(S).read()
m=open(M).read()
a=open(A).read()

def replace_once(text, old, new, name):
    assert old in text, f"ANCHOR NOT FOUND: {name}"
    assert new not in text, f"ALREADY PATCHED: {name}"
    return text.replace(old,new,1)

# ------------------------------------------------------------------
# 1) Exact R1 edit-type routing used in the previous combined model,
#    plus a per-case switch telling M-HFF which edit types are active.
# ------------------------------------------------------------------
router_old = "        edit_param = case['edit_param']"
router_new = '''        edit_param = case['edit_param']
        _dx,_dy,_dz,_rx,_ry,_rz,_sx,_sy,_sz = edit_param
        if abs(float(_rz))>1e-6:
            _etype='rotate'
        elif abs(float(_sx)-1)>1e-6 or abs(float(_sy)-1)>1e-6:
            _etype='resize'
        else:
            _etype='move'

        if os.environ.get('FF_ROUTER','0')=='1':
            for _k in ('FF_PRESERVE','FF_PRESERVE_W','FF_USE_PROMPT','FF_GUIDANCE'):
                os.environ.pop(_k,None)
            if _etype=='move':
                os.environ['FF_PRESERVE']='1'
                os.environ['FF_PRESERVE_W']='0.3'
            elif _etype=='resize':
                os.environ['FF_USE_PROMPT']='1'
                os.environ['FF_GUIDANCE']='10'
            # rotate remains exact baseline.

        _ff_types={x.strip() for x in os.environ.get('FF_HFF_TYPES','').split(',') if x.strip()}
        os.environ['FF_HFF_ACTIVE'] = (
            '1' if os.environ.get('FF_HFF','0')=='1' and _etype in _ff_types else '0'
        )'''
s=replace_once(s,router_old,router_new,"router + HFF active")

# ------------------------------------------------------------------
# 2) Same move latent-preservation implementation as previous R1.
# ------------------------------------------------------------------
pres_old = (
"                    latents = self.ctrl_step(noise_pred, t, latents, local_var_reg, eta=eta)[0]\n"
"                latents_list.append(latents)"
)
pres_new = (
"                    latents = self.ctrl_step(noise_pred, t, latents, local_var_reg, eta=eta)[0]\n"
"                if os.environ.get('FF_PRESERVE')=='1' and latents.shape[0]==2:\n"
"                    _cl = refer_latents[i - start_step + 1][0]\n"
"                    _w = float(os.environ.get('FF_PRESERVE_W','0.3'))\n"
"                    _m = local_var_reg[0].to(latents.dtype) if local_var_reg.dim()==4 else local_var_reg.to(latents.dtype)\n"
"                    latents[0] = latents[0]*(1 - _w*_m) + _cl*(_w*_m)\n"
"                latents_list.append(latents)"
)
assert pres_old in m, "preservation anchor missing"
m=m.replace(pres_old,pres_new,1)

# ------------------------------------------------------------------
# 3) Extend overridden UNet forward with feature fusion after a decoder block.
# ------------------------------------------------------------------
sig_old = "        last_up_block_idx: int = None,\n    ):"
sig_new = '''        last_up_block_idx: int = None,
        ff_feature_ref = None,
        ff_feature_mask = None,
        ff_feature_beta: float = 0.0,
        ff_feature_mode: str = "hf",
        ff_feature_radius: int = 3,
        ff_feature_block: int = 1,
    ):'''
a=replace_once(a,sig_old,sig_new,"UNet feature-fusion signature")

# Robust insertion point: upstream FreeFine revisions may change/remove
# the comment following all_intermediate_features.append(sample).
_fuse_pat = re.compile(
    r"(?m)^(?P<indent>[ \t]*)all_intermediate_features\.append\(sample\)[ \t]*$"
)
_fuse_matches = list(_fuse_pat.finditer(a))
assert len(_fuse_matches) == 1, (
    "M-HFF insertion point expected exactly once, "
    f"found {len(_fuse_matches)} occurrences"
)
_fm = _fuse_matches[0]
_indent = _fm.group('indent')

_body_lines = [
    "# Optional M-HFF / full-feature residual fusion.",
    "# Batch semantics: [uncond_edit, uncond_ref, cond_edit, cond_ref].",
    "if ff_feature_ref is not None and i == int(ff_feature_block) and float(ff_feature_beta) > 0:",
    "    _ref = ff_feature_ref.to(device=sample.device, dtype=sample.dtype)",
    "    if _ref.shape != sample.shape:",
    "        raise RuntimeError(f'M-HFF feature shape mismatch: ref={_ref.shape}, cur={sample.shape}')",
    "",
    "    if ff_feature_mask is None:",
    "        _mask = torch.ones((sample.shape[0],1,*sample.shape[-2:]), device=sample.device, dtype=sample.dtype)",
    "    else:",
    "        _mask = ff_feature_mask",
    "        if _mask.dim()==2:",
    "            _mask=_mask[None,None]",
    "        elif _mask.dim()==3:",
    "            _mask=_mask[:,None]",
    "        _mask = F.interpolate(_mask.float(), size=sample.shape[-2:], mode='bilinear', align_corners=False).to(device=sample.device,dtype=sample.dtype)",
    "        if _mask.shape[0] != sample.shape[0]:",
    "            _mask = _mask[:1].expand(sample.shape[0],-1,-1,-1)",
    "",
    "    # Never alter source-reference streams.",
    "    _gate=torch.zeros((sample.shape[0],1,1,1),device=sample.device,dtype=sample.dtype)",
    "    if sample.shape[0] >= 4:",
    "        _gate[0]=1",
    "        _gate[2]=1",
    "    else:",
    "        _gate[0]=1",
    "    _mask = _mask * _gate",
    "",
    "    if str(ff_feature_mode).lower() == 'hf':",
    "        def _high_pass(_x, _r):",
    "            _xf=torch.fft.fftshift(torch.fft.fft2(_x.float(),dim=(-2,-1)), dim=(-2,-1))",
    "            _h,_w=_x.shape[-2:]",
    "            _yy=torch.arange(_h,device=_x.device)[:,None]",
    "            _xx=torch.arange(_w,device=_x.device)[None,:]",
    "            _cy,_cx=_h//2,_w//2",
    "            _keep=((_yy-_cy)**2+(_xx-_cx)**2 > int(_r)**2).float()[None,None]",
    "            _yf=_xf*_keep",
    "            _y=torch.fft.ifft2(torch.fft.ifftshift(_yf,dim=(-2,-1)), dim=(-2,-1)).real",
    "            return _y.to(dtype=_x.dtype)",
    "        _cur_part=_high_pass(sample,ff_feature_radius)",
    "        _ref_part=_high_pass(_ref,ff_feature_radius)",
    "        _delta=_ref_part-_cur_part",
    "    elif str(ff_feature_mode).lower() == 'full':",
    "        _delta=_ref-sample",
    "    else:",
    "        raise ValueError(f'Unknown ff_feature_mode={ff_feature_mode}')",
    "",
    "    sample = sample + float(ff_feature_beta) * _mask * _delta",
]
_fuse_insert = ''.join(_indent + line + '\n' for line in _body_lines)
a = a[:_fm.start()] + _fuse_insert + a[_fm.start():]
print(f"✓ M-HFF insertion point found; indent={len(_indent)}")

# ------------------------------------------------------------------
# 4) Reference feature = SAME-timestep feature from already available
#    coarse DDIM trajectory. No extra model and no CLIP compression.
# ------------------------------------------------------------------
assert "    def forward_sampling(" in m and "    def prox_regularization" in m
pre, rest = m.split("    def forward_sampling(",1)
body, post = rest.split("    def prox_regularization",1)

noise_new = '''            self.controller.log_mask = False

            _ff_ref = None
            _ff_beta = 0.0
            if os.environ.get('FF_HFF_ACTIVE','0')=='1':
                _denom=max(1.0,float((num_inference_steps-1)-start_step))
                _progress=float(i-start_step)/_denom
                _horizon=float(os.environ.get('FF_HFF_HORIZON','0.55'))
                _beta0=float(os.environ.get('FF_HFF_BETA','0.30'))
                if _horizon > 0 and _progress <= _horizon:
                    _ff_beta=_beta0*0.5*(1.0+np.cos(np.pi*_progress/_horizon))

                if _ff_beta > 1e-8:
                    _ri=i-start_step+1
                    _coarse_lat=refer_latents[_ri][0:1]
                    _src_lat=refer_latents[_ri][1:2]
                    _ref_pair=torch.cat([_coarse_lat,_src_lat],dim=0)
                    _ref_inputs=torch.cat([_ref_pair]*2,dim=0)

                    # Partial reference forward must not advance TCA bookkeeping.
                    _saved_att=getattr(self.controller,'cur_att_layer',None)
                    _saved_step=getattr(self.controller,'cur_step',None)
                    _ff_list=self.unet(
                        _ref_inputs, t,
                        encoder_hidden_states=text_embeddings,
                        last_up_block_idx=int(os.environ.get('FF_HFF_BLOCK','1'))
                    )
                    if _saved_att is not None:
                        self.controller.cur_att_layer=_saved_att
                    if _saved_step is not None:
                        self.controller.cur_step=_saved_step
                    _ff_ref=_ff_list[-1].detach()

            noise_pred = self.unet(
                model_inputs, t,
                encoder_hidden_states=text_embeddings,
                ff_feature_ref=_ff_ref,
                ff_feature_mask=getattr(self.controller,'fg_retain_mask_st2',None),
                ff_feature_beta=_ff_beta,
                ff_feature_mode=os.environ.get('FF_HFF_MODE','hf'),
                ff_feature_radius=int(os.environ.get('FF_HFF_RADIUS','3')),
                ff_feature_block=int(os.environ.get('FF_HFF_BLOCK','1')),
            )'''

# --- FIX: match the UNet call by regex and re-indent the replacement to the real depth.
#     Upstream is 16 spaces (inside `with torch.no_grad():`); the block above is authored at 12.
_noise_pat = re.compile(
    r"(?m)^(?P<indent>[ \t]+)self\.controller\.log_mask = False[ \t]*\n"
    r"(?P=indent)noise_pred = self\.unet\(model_inputs, t, encoder_hidden_states=text_embeddings\)[ \t]*$"
)
_nm = list(_noise_pat.finditer(body))
assert len(_nm) == 1, f"forward_sampling UNet anchor expected exactly once; found {len(_nm)}"
_nindent = _nm[0].group("indent")
_nlines = noise_new.split("\n")
_nmin = min(len(l) - len(l.lstrip(" ")) for l in _nlines if l.strip())
noise_new = "\n".join((_nindent + l[_nmin:]) if l.strip() else "" for l in _nlines)
body = body[:_nm[0].start()] + noise_new + body[_nm[0].end():]
print(f"✓ forward_sampling UNet anchor found; indent={len(_nindent)}")

m=pre+"    def forward_sampling("+body+"    def prox_regularization"+post

open(S,"w").write(s)
open(M,"w").write(m)
open(A,"w").write(a)

assert "FF_HFF_ACTIVE" in open(S).read()
assert "ff_feature_ref" in open(A).read()
assert "FF_HFF_BETA" in open(M).read()
import py_compile
for _f in (S,M,A):
    py_compile.compile(_f,doraise=True)
print("✓ Account A patches installed + syntax-compiled: R1 router + M-HFF/full-feature fusion")

✓ M-HFF insertion point found; indent=12
✓ forward_sampling UNet anchor found; indent=16
✓ Account A patches installed + syntax-compiled: R1 router + M-HFF/full-feature fusion


## 7. Dataset — exact historical balanced-200 + resize/rotate manifests

In [7]:

# ===== C6 data + EXACT balanced-200 used in prior experiments =====
import os, glob, json, csv, random, shutil, hashlib
from collections import defaultdict, Counter

GEO="/kaggle/temp/GeoBenchMeta"
os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)

CACHE=next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True)
           if os.path.isdir(f"{c}/source_img"))
COARSE=glob.glob("/kaggle/input/**/coarse_img/*/*/*.png",recursive=True)[0].split("/coarse_img/")[0]+"/coarse_img"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
IB=os.path.dirname(os.path.dirname(os.path.dirname(
    glob.glob("/kaggle/input/**/inp_img_blended/**/inp_img.png",recursive=True)[0])))
ANNs=glob.glob("/kaggle/input/**/annotation_2d.json",recursive=True)[0]
META=glob.glob("/kaggle/input/**/sample_metadata.csv",recursive=True)[0]

for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if not os.path.exists(d):
        os.symlink(f"{CACHE}/{nm}",d)

for nm,sc in [("coarse_img",COARSE),("inp_img_blended",IB)]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if not os.path.exists(d):
        os.symlink(sc,d)

shutil.copy(ANNs,f"{GEO}/annotation_2d.json")
ann=json.load(open(f"{GEO}/annotation_2d.json"))
meta=[r for r in csv.DictReader(open(META))
      if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]

# IMPORTANT: identical selection algorithm and seed as all previous balanced-200 studies.
random.seed(42)
cells=defaultdict(list)
for r in meta:
    cells[(r["edit_type"],r["difficulty"])].append(r)

keys=sorted(cells)
per=200//len(keys)
picked=[]
for k in keys:
    pool=cells[k][:]
    random.shuffle(pool)
    picked += pool[:per]

chosen={(r["da_n"],r["ins_id"],r["case_id"]) for r in picked}
left=[r for r in meta if (r["da_n"],r["ins_id"],r["case_id"]) not in chosen]
random.shuffle(left)
for r in left:
    if len(picked)>=200:
        break
    picked.append(r)
picked=picked[:200]

counts=Counter(r["edit_type"] for r in picked)
print("balanced-200:",len(picked),dict(counts))
assert len(picked)==200
assert counts["move"]==67 and counts["resize"]==67 and counts["rotate"]==66, \
    f"Unexpected subset counts: {counts}. Check that the same GeoBench metadata is mounted."

json.dump(picked,open(f"{GEO}/subset_meta.json","w"),indent=2)

# Cross-account provenance fingerprint. Account A and B MUST print the same hash.
fp_rows=sorted(f"{r['da_n']}|{r['ins_id']}|{r['case_id']}" for r in picked)
SUBSET_SHA256=hashlib.sha256("\n".join(fp_rows).encode()).hexdigest()
print("balanced-200 SHA256:",SUBSET_SHA256)

def build_generation_manifest(rows, out_json):
    gsub={}
    for r in rows:
        d,i,e=r["da_n"],r["ins_id"],r["case_id"]
        lf=dict(ann[d]["instances"][i][e])
        lf["ori_img_path"]=os.path.join(GEO,lf["ori_img_path"])
        lf["ori_mask_path"]=os.path.join(GEO,lf["ori_mask_path"])
        gsub.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
    json.dump(gsub,open(out_json,"w"))
    return out_json

build_generation_manifest(picked, f"{GEO}/gen_subset.json")
resize_rows=[r for r in picked if r["edit_type"]=="resize"]
rotate_rows=[r for r in picked if r["edit_type"]=="rotate"]
move_rows=[r for r in picked if r["edit_type"]=="move"]
build_generation_manifest(resize_rows, f"{GEO}/gen_resize.json")
build_generation_manifest(rotate_rows, f"{GEO}/gen_rotate.json")
build_generation_manifest(move_rows, f"{GEO}/gen_move.json")
json.dump(resize_rows,open(f"{GEO}/subset_meta_resize.json","w"),indent=2)
json.dump(rotate_rows,open(f"{GEO}/subset_meta_rotate.json","w"),indent=2)

os.makedirs(f"{GEO}/gen_eval",exist_ok=True)
d=f"{GEO}/gen_eval/baseline"
if os.path.islink(d):
    os.remove(d)
elif os.path.exists(d):
    shutil.rmtree(d)
os.symlink(GENBASE,d)
print("baseline eval linked:",GENBASE)


balanced-200: 200 {'move': 67, 'resize': 67, 'rotate': 66}
balanced-200 SHA256: 3d7c0172cba1e35693a3f20b562004bcfd6fc43e187160c945a23224d40cbd3c
baseline eval linked: /kaggle/input/datasets/georgiostzamouranis/freefine-geobench2d-bggen/gen_results_2d_final/gen_results_2d_backup


## 8. Baseline validation gate — method OFF must reproduce stored baseline

In [8]:
# ===== C7 validation gate (defaults must reproduce baseline) =====
import os, json, socket, subprocess, glob, numpy as np
from PIL import Image
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"]=UserSecretsClient().get_secret("HF_TOKEN")
GEO="/kaggle/temp/GeoBenchMeta"; P="/kaggle/temp/FreeFine/evaluation/FreeFine"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
gs=json.load(open(f"{GEO}/gen_subset.json")); val={}; n=0
for d,da in gs.items():
    for i,ins in da["instances"].items():
        for e in ins:
            val.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=ins[e]; n+=1
            if n>=5:break
        if n>=5:break
    if n>=5:break
json.dump(val,open(f"{GEO}/val5.json","w")); os.makedirs("/kaggle/temp/val5",exist_ok=True)
env=os.environ.copy(); env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/val5.json","FF_OUT_DIR":"/kaggle/temp/val5"})
s=socket.socket();s.bind(("",0));port=s.getsockname()[1];s.close()
r=subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1","--master-port",str(port),"freefine_sweep_2d.py"],cwd=P,env=env,capture_output=True,text=True)
imgs=sorted(glob.glob("/kaggle/temp/val5/**/*.png",recursive=True))
if not imgs: print((r.stdout+r.stderr)[-3000:]); raise SystemExit("validation 0 images")
diffs=[float(np.abs(np.array(Image.open(n).convert("RGB"),float)-np.array(Image.open(f"{GENBASE}/{os.path.relpath(n,'/kaggle/temp/val5')}").convert("RGB").resize(Image.open(n).size),float)).mean()) for n in imgs]
print("validation mean|Δ|:",[round(x,3) for x in diffs]); assert max(diffs)<1.0,"NOT REPRODUCING"; print("✓ validation passed")

validation mean|Δ|: [0.0, 0.0, 0.0, 0.0, 0.0]
✓ validation passed


## 9. Parallel T4×2 generation scheduler

In [9]:

# ===== C8A parallel generation on T4x2 =====
import os, time, glob, subprocess, socket, json, shutil
from collections import deque

GEO="/kaggle/temp/GeoBenchMeta"
P="/kaggle/temp/FreeFine/evaluation/FreeFine"
OUT="/kaggle/working/accountA_feature_discovery/variants"
os.makedirs(OUT,exist_ok=True)

GEN_SOFT_DEADLINE=NB_START+8.8*3600
GEN_HARD_STOP=NB_START+9.5*3600

# Same R1 resize reference + five resize feature experiments + one rotate experiment.
JOBS=[
    ("R1_resize", f"{GEO}/gen_resize.json", 67,
     {"FF_ROUTER":1}),
    ("HF15", f"{GEO}/gen_resize.json", 67,
     {"FF_ROUTER":1,"FF_HFF":1,"FF_HFF_TYPES":"resize","FF_HFF_MODE":"hf",
      "FF_HFF_BETA":0.15,"FF_HFF_HORIZON":0.55,"FF_HFF_RADIUS":3,"FF_HFF_BLOCK":1}),
    ("HF30", f"{GEO}/gen_resize.json", 67,
     {"FF_ROUTER":1,"FF_HFF":1,"FF_HFF_TYPES":"resize","FF_HFF_MODE":"hf",
      "FF_HFF_BETA":0.30,"FF_HFF_HORIZON":0.55,"FF_HFF_RADIUS":3,"FF_HFF_BLOCK":1}),
    ("HF50", f"{GEO}/gen_resize.json", 67,
     {"FF_ROUTER":1,"FF_HFF":1,"FF_HFF_TYPES":"resize","FF_HFF_MODE":"hf",
      "FF_HFF_BETA":0.50,"FF_HFF_HORIZON":0.55,"FF_HFF_RADIUS":3,"FF_HFF_BLOCK":1}),
    ("FULL30", f"{GEO}/gen_resize.json", 67,
     {"FF_ROUTER":1,"FF_HFF":1,"FF_HFF_TYPES":"resize","FF_HFF_MODE":"full",
      "FF_HFF_BETA":0.30,"FF_HFF_HORIZON":0.55,"FF_HFF_RADIUS":3,"FF_HFF_BLOCK":1}),
    ("HF30_E35", f"{GEO}/gen_resize.json", 67,
     {"FF_ROUTER":1,"FF_HFF":1,"FF_HFF_TYPES":"resize","FF_HFF_MODE":"hf",
      "FF_HFF_BETA":0.30,"FF_HFF_HORIZON":0.35,"FF_HFF_RADIUS":3,"FF_HFF_BLOCK":1}),
    ("ROT_HF30", f"{GEO}/gen_rotate.json", 66,
     {"FF_ROUTER":1,"FF_HFF":1,"FF_HFF_TYPES":"rotate","FF_HFF_MODE":"hf",
      "FF_HFF_BETA":0.30,"FF_HFF_HORIZON":0.45,"FF_HFF_RADIUS":3,"FF_HFF_BLOCK":1}),
]

json.dump(
    [{"tag":t,"subset":os.path.basename(sub),"expected":exp,"env":opts} for t,sub,exp,opts in JOBS],
    open("/kaggle/working/accountA_experiment_manifest.json","w"),indent=2
)

def img_count(path):
    return len(glob.glob(path+"/**/*.png",recursive=True))

def free_port():
    s=socket.socket()
    s.bind(("",0))
    p=s.getsockname()[1]
    s.close()
    return p

def launch(tag, subset, expected, opts, gpu):
    out=f"{OUT}/{tag}"
    os.makedirs(out,exist_ok=True)
    env=os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES":str(gpu),
        "PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],
        "PYTHONUNBUFFERED":"1",
        "TOKENIZERS_PARALLELISM":"false",
        "HF_HOME":"/kaggle/temp/hf",
        "NCCL_P2P_DISABLE":"1",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "FF_SUBSET_JSON":subset,
        "FF_OUT_DIR":out,
    })
    env.update({k:str(v) for k,v in opts.items()})
    log=open(f"{out}/log.txt","w")
    cmd=[
        "/kaggle/temp/freefine_env/bin/torchrun",
        "--nproc_per_node=1",
        "--master-port",str(free_port()),
        "freefine_sweep_2d.py"
    ]
    proc=subprocess.Popen(cmd,cwd=P,env=env,stdout=log,stderr=subprocess.STDOUT)
    return {"tag":tag,"subset":subset,"expected":expected,"opts":opts,
            "gpu":gpu,"proc":proc,"log":log,"start":time.time(),"out":out}

queue=deque()
for job in JOBS:
    tag,subset,expected,opts=job
    if img_count(f"{OUT}/{tag}")>=expected:
        print("already complete:",tag)
    else:
        queue.append(job)

running={}
done=[]
failed=[]

while queue or running:
    # Finish completed processes.
    for gpu,info in list(running.items()):
        rc=info["proc"].poll()
        if rc is not None:
            info["log"].close()
            n=img_count(info["out"])
            dt=(time.time()-info["start"])/60
            print(f"[GPU{gpu}] {info['tag']} finished rc={rc} images={n}/{info['expected']} time={dt:.1f}m",flush=True)
            if rc==0 and n>=info["expected"]:
                done.append(info["tag"])
            else:
                failed.append(info["tag"])
                try:
                    tail=open(f"{info['out']}/log.txt").read()[-2500:]
                    print("LOG TAIL:\n",tail,flush=True)
                except Exception:
                    pass
            del running[gpu]

    # Hard stop protects Save & Run All from the 12h limit.
    if time.time()>GEN_HARD_STOP and running:
        print("GEN_HARD_STOP: terminating remaining jobs",flush=True)
        for gpu,info in list(running.items()):
            info["proc"].terminate()
        time.sleep(10)
        for gpu,info in list(running.items()):
            if info["proc"].poll() is None:
                info["proc"].kill()
            info["log"].close()
            failed.append(info["tag"])
            del running[gpu]
        break

    # Fill free GPUs while before soft deadline.
    for gpu in [0,1]:
        if gpu not in running and queue and time.time()<GEN_SOFT_DEADLINE:
            tag,subset,expected,opts=queue.popleft()
            running[gpu]=launch(tag,subset,expected,opts,gpu)
            print(f"[GPU{gpu}] launched {tag}",flush=True)

    if queue and not running and time.time()>=GEN_SOFT_DEADLINE:
        print("GEN_SOFT_DEADLINE: not launching:",[j[0] for j in queue],flush=True)
        break

    time.sleep(15)

status={"done":done,"failed":failed,"not_launched":[j[0] for j in queue]}
json.dump(status,open("/kaggle/working/accountA_generation_status.json","w"),indent=2)
print("generation status:",status)


[GPU0] launched R1_resize
[GPU1] launched HF15
[GPU0] R1_resize finished rc=0 images=67/67 time=45.8m
[GPU0] launched HF30
[GPU1] HF15 finished rc=0 images=67/67 time=50.5m
[GPU1] launched HF50
[GPU0] HF30 finished rc=0 images=67/67 time=51.5m
[GPU0] launched FULL30
[GPU1] HF50 finished rc=0 images=67/67 time=50.8m
[GPU1] launched HF30_E35
[GPU0] FULL30 finished rc=0 images=67/67 time=51.8m
[GPU0] launched ROT_HF30
[GPU1] HF30_E35 finished rc=0 images=67/67 time=48.5m
[GPU0] ROT_HF30 finished rc=0 images=66/66 time=50.3m
generation status: {'done': ['R1_resize', 'HF15', 'HF30', 'HF50', 'FULL30', 'HF30_E35', 'ROT_HF30'], 'failed': [], 'not_launched': []}


## 10. Evaluation on the exact historical groups

In [10]:

# ===== C9A evaluation: SAME group definitions + SAME seeded MD as prior experiments =====
import os, json, re, glob, subprocess, time, numpy as np, shutil, csv
from PIL import Image

GEO="/kaggle/temp/GeoBenchMeta"
MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"
OUT="/kaggle/working/accountA_feature_discovery/variants"
EVAL_DEADLINE=NB_START+11.25*3600

ann=json.load(open(f"{GEO}/annotation_2d.json"))
picked=json.load(open(f"{GEO}/subset_meta.json"))

# Link every completed generated arm into the exact same evaluation structure.
for tag in ["R1_resize","HF15","HF30","HF50","FULL30","HF30_E35","ROT_HF30"]:
    p=f"{OUT}/{tag}"
    if glob.glob(p+"/**/*.png",recursive=True):
        d=f"{GEO}/gen_eval/{tag}"
        if os.path.islink(d):
            os.remove(d)
        elif os.path.exists(d):
            shutil.rmtree(d)
        os.symlink(p,d)

def mem(pred):
    return [(r["da_n"],r["ins_id"],r["case_id"]) for r in picked if pred(r)]

groups={
    "resize_all":mem(lambda r:r["edit_type"]=="resize"),
    "resize_hard":mem(lambda r:r["edit_type"]=="resize" and r["difficulty"]=="hard"),
    "rotate_all":mem(lambda r:r["edit_type"]=="rotate"),
    "rotate_hard":mem(lambda r:r["edit_type"]=="rotate" and r["difficulty"]=="hard"),
}

def wrap_e(ids,gd):
    tot=0.0
    n=0
    for d,i,e in ids:
        cp=f"{GEO}/Geo-Bench-2D/coarse_img/{d}/{i}/{e}.png"
        gp=f"{gd}/{d}/{i}/{e}.png"
        tp=f"{GEO}/Geo-Bench-2D/target_mask/{d}/{i}/{e}.png"
        if not all(os.path.exists(x) for x in (cp,gp,tp)):
            continue
        C=np.array(Image.open(cp).convert("RGB"),float)/255.0
        G=np.array(Image.open(gp).convert("RGB"),float)/255.0
        T=np.array(Image.open(tp).convert("L"),float)/255.0
        if G.shape[:2]!=C.shape[:2]:
            G=np.array(Image.fromarray((G*255).astype("uint8")).resize((C.shape[1],C.shape[0])),float)/255.0
        if T.shape[:2]!=C.shape[:2]:
            T=np.array(Image.fromarray((T*255).astype("uint8")).resize((C.shape[1],C.shape[0])),float)/255.0
        mm=np.repeat(T[...,None],3,axis=2)
        su=mm.sum()
        if su<=0:
            continue
        tot+=float(np.sum(np.abs(C*mm-G*mm))/su)
        n+=1
    return round(tot/n,4) if n else None

def manifest(setn,ids):
    o={}
    b=f"{GEO}/gen_eval/{setn}"
    used=0
    for d,i,e in ids:
        if not os.path.exists(f"{b}/{d}/{i}/{e}.png"):
            continue
        lf=dict(ann[d]["instances"][i][e])
        lf["gen_img_path"]=f"gen_eval/{setn}/{d}/{i}/{e}.png"
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
        used+=1
    pth=f"{GEO}/m_{setn}_{len(ids)}.json"
    json.dump(o,open(pth,"w"))
    return pth,used

def run_metrics(manp):
    env=os.environ.copy()
    env.update({
        "MPLBACKEND":"Agg",
        "HF_HOME":"/kaggle/temp/hf",
        "TORCH_HOME":"/kaggle/temp/torch",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True"
    })
    # Same metric task mask used for group studies: SUBC + BGC + MD.
    out=subprocess.run(
        [PY,"main.py","--path",manp,"--use_relative_path",
         "--base_dir",GEO,
         "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2",
         "--task","000110100","--level","0"],
        cwd=MET,env=env,capture_output=True,text=True
    )
    txt=out.stdout+out.stderr
    vals={}
    for k in ["SUBC","BGC","MD"]:
        hits=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",txt)
        if hits:
            vals[k]=round(float(hits[-1]),4)
    if out.returncode!=0:
        vals["_rc"]=out.returncode
        vals["_tail"]=txt[-1200:]
    return vals

# Baseline is evaluated on the same groups. Resize feature arms compare to R1_resize;
# ROT_HF30 compares to baseline because R1 leaves rotation untouched.
sets=["baseline","R1_resize","HF15","HF30","HF50","FULL30","HF30_E35","ROT_HF30"]
results={}

for setn in sets:
    if time.time()>EVAL_DEADLINE:
        print("EVAL_DEADLINE reached",flush=True)
        break
    if not os.path.exists(f"{GEO}/gen_eval/{setn}"):
        continue

    if setn=="ROT_HF30":
        wanted=["rotate_all","rotate_hard"]
    elif setn=="baseline":
        wanted=["resize_all","resize_hard","rotate_all","rotate_hard"]
    else:
        wanted=["resize_all","resize_hard"]

    results[setn]={}
    for g in wanted:
        if time.time()>EVAL_DEADLINE:
            break
        manp,used=manifest(setn,groups[g])
        if used==0:
            continue
        v=run_metrics(manp)
        v["WRAP_E"]=wrap_e(groups[g],f"{GEO}/gen_eval/{setn}")
        v["n"]=used
        results[setn][g]=v
        print(f"{setn:10s} {g:12s} -> {v}",flush=True)
        json.dump(results,open("/kaggle/working/accountA_feature_results.json","w"),indent=2)

# Flat CSV for easy merge with previous experiment tables.
rows=[]
for s,gd in results.items():
    for g,v in gd.items():
        rows.append({"set":s,"group":g,**{k:v.get(k) for k in ["SUBC","BGC","WRAP_E","MD","n"]}})
with open("/kaggle/working/accountA_feature_results.csv","w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=["set","group","SUBC","BGC","WRAP_E","MD","n"])
    w.writeheader()
    w.writerows(rows)

print("saved Account A results")


baseline   resize_all   -> {'_rc': 1, '_tail': '", line 59, in _download\n    with urllib.request.urlopen(url) as source, open(download_target, "wb") as output:\n  File "/root/.local/share/uv/python/cpython-3.10.13-linux-x86_64-gnu/lib/python3.10/urllib/request.py", line 216, in urlopen\n    return opener.open(url, data, timeout)\n  File "/root/.local/share/uv/python/cpython-3.10.13-linux-x86_64-gnu/lib/python3.10/urllib/request.py", line 519, in open\n    response = self._open(req, data)\n  File "/root/.local/share/uv/python/cpython-3.10.13-linux-x86_64-gnu/lib/python3.10/urllib/request.py", line 536, in _open\n    result = self._call_chain(self.handle_open, protocol, protocol +\n  File "/root/.local/share/uv/python/cpython-3.10.13-linux-x86_64-gnu/lib/python3.10/urllib/request.py", line 496, in _call_chain\n    result = func(*args)\n  File "/root/.local/share/uv/python/cpython-3.10.13-linux-x86_64-gnu/lib/python3.10/urllib/request.py", line 1391, in https_open\n    return self.do_ope

## 11. Summary / promotion decisions

In [11]:

# ===== C10A compact comparison =====
import json
R=json.load(open("/kaggle/working/accountA_feature_results.json"))

print("\nACCOUNT A — FEATURE/FREQUENCY DISCOVERY")
print("Primary comparison: resize arms vs R1_resize; ROT_HF30 vs baseline.")
print("All groups are the exact historical balanced-200 groups; MD is seeded 42.\n")

for g in ["resize_all","resize_hard","rotate_all","rotate_hard"]:
    print(f"\n=== {g} ===")
    print(f"{'set':12s} {'SUBC':>9s} {'BGC':>9s} {'WRAP_E':>9s} {'MD':>9s}")
    for s in ["baseline","R1_resize","HF15","HF30","HF50","FULL30","HF30_E35","ROT_HF30"]:
        if s in R and g in R[s]:
            d=R[s][g]
            print(f"{s:12s} {str(d.get('SUBC','-')):>9s} {str(d.get('BGC','-')):>9s} "
                  f"{str(d.get('WRAP_E','-')):>9s} {str(d.get('MD','-')):>9s}")

def delta(tag,ref,group):
    if tag not in R or ref not in R or group not in R[tag] or group not in R[ref]:
        return None
    a,b=R[tag][group],R[ref][group]
    return {k:round(a[k]-b[k],4) for k in ["SUBC","BGC","WRAP_E","MD"] if k in a and k in b}

print("\nΔ vs R1_resize (negative MD/WRAP_E is better; positive SUBC/BGC is better):")
for s in ["HF15","HF30","HF50","FULL30","HF30_E35"]:
    print(s,delta(s,"R1_resize","resize_hard"))

print("\nROT_HF30 Δ vs exact baseline:")
print(delta("ROT_HF30","baseline","rotate_hard"))

print("\nSelection rule for next synthesis run:")
print("Prefer a robust MD decrease (>=~1.0 on hard group or >=5%) with no material SUBC/BGC regression.")
print("FULL30 is an ablation: HF > FULL supports frequency-selective transfer rather than generic feature copying.")
print("FID/DINO/KD intentionally deferred to the next full-200 synthesis run; this discovery run uses the same")
print("group-level SUBC/BGC/WRAP_E/seeded-MD protocol used in the earlier hard-group studies.")



ACCOUNT A — FEATURE/FREQUENCY DISCOVERY
Primary comparison: resize arms vs R1_resize; ROT_HF30 vs baseline.
All groups are the exact historical balanced-200 groups; MD is seeded 42.


=== resize_all ===
set               SUBC       BGC    WRAP_E        MD
baseline             -         -    0.0544         -
R1_resize       0.8908    0.9642    0.0547   11.2776
HF15            0.8908    0.9642    0.0547   11.2828
HF30            0.8908    0.9642    0.0547   11.2684
HF50            0.8908    0.9642    0.0547   11.4942
FULL30          0.8908    0.9642    0.0547   11.2632
HF30_E35        0.8908    0.9643    0.0547   11.2774

=== resize_hard ===
set               SUBC       BGC    WRAP_E        MD
baseline        0.8391    0.9631    0.0585   19.4713
R1_resize       0.8391    0.9625    0.0587   17.7459
HF15            0.8392    0.9626    0.0587   17.7956
HF30            0.8392    0.9626    0.0587    17.028
HF50            0.8392    0.9626    0.0586   18.0082
FULL30          0.8393    0.9625 